# LiViFuser official DINOv3 S+/16 backbone handoff v1

Accept access to the gated Hugging Face model in your browser, add a private Kaggle secret named `HF_TOKEN`, enable Internet, and run all cells. This notebook downloads no policy data and performs no training or held-out inference.


In [ ]:
%pip install -q huggingface_hub==0.34.4
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
if not HF_TOKEN or not HF_TOKEN.strip():
    raise RuntimeError('HF_TOKEN is missing or empty')


In [ ]:
# Embedded source SHA-256: F6FFE986AD2D4EE67D86F9C3AFA228AA665F10D5AA5083A82D22615B44C4B09B
CORE_SOURCE = '"""Deterministic acquisition and sealing of the locked DINOv3 S+/16 snapshot."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport zipfile\nfrom pathlib import Path\nfrom typing import Any\n\nMODEL_ID = "facebook/dinov3-vits16plus-pretrain-lvd1689m"\nMODEL_REVISION = "c93d816fc9e567563bc068f01475bec89cc634a6"\nBUNDLE_ROOT = "livifuser_dinov3_vits16plus_backbone_c93d816"\nBUNDLE_FILENAME = f"{BUNDLE_ROOT}_bundle.zip"\nMANIFEST_NAME = "BACKBONE_BUNDLE_MANIFEST.json"\nCOMPLETE_NAME = "BACKBONE_BUNDLE_COMPLETE.json"\nZIP_TIMESTAMP = (1980, 1, 1, 0, 0, 0)\n\n# This exact set was independently recorded by the accepted train/validation\n# DINO cache contract. Do not discover or widen it from a future Hub snapshot.\nEXPECTED_MODEL_FILES: dict[str, dict[str, Any]] = {\n    "LICENSE.md": {\n        "size_bytes": 7_503,\n        "sha256": "25D122EB8F5B880FD23C736FB6EA8018EE45C12237E00B8A86D14C653904999E",\n    },\n    "README.md": {\n        "size_bytes": 14_528,\n        "sha256": "75CD3E334E64FECFA2507B4ECE416964D4BCAFCDED94D8B351D00679B95A5B5D",\n    },\n    "config.json": {\n        "size_bytes": 742,\n        "sha256": "6F4AC67FEA1761FE684D2A7DB3139BAB2D0DFDF94C05063D5992717C4C1DA0AC",\n    },\n    "model.safetensors": {\n        "size_bytes": 114_794_096,\n        "sha256": "208146E499DACE99E4C9376DDB8A26F77D64C31C46C4DC4B86FF8BC63B0235E2",\n    },\n    "preprocessor_config.json": {\n        "size_bytes": 585,\n        "sha256": "960C41D1F3A7778B936365769A2D90550B318A6C0A53A0296957ADACFE5E0DD7",\n    },\n}\n\nACCEPTED_CACHE_BACKBONE_CONTRACT_FILE_SHA256 = (\n    "2957C78346DE608067DD5AC14D5C3E2F23438CD2BB3B1ECA847F898EBA68894A"\n)\nACCEPTED_CACHE_BACKBONE_CONTRACT_SELF_SHA256 = (\n    "DA76FCBC0A0309DB6F98C742924CF579827123217B6B9410D74F83FC6AC0D772"\n)\n\n\ndef sha256_bytes(payload: bytes) -> str:\n    return hashlib.sha256(payload).hexdigest().upper()\n\n\ndef sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        while chunk := stream.read(chunk_size):\n            digest.update(chunk)\n    return digest.hexdigest().upper()\n\n\ndef json_bytes(value: object) -> bytes:\n    return (\n        json.dumps(value, indent=2, sort_keys=True, ensure_ascii=False).encode("utf-8")\n        + b"\\n"\n    )\n\n\ndef self_hash(value: dict[str, Any], field: str) -> str:\n    copy = dict(value)\n    copy.pop(field, None)\n    return sha256_bytes(json_bytes(copy))\n\n\ndef _zip_info(name: str) -> zipfile.ZipInfo:\n    info = zipfile.ZipInfo(name, date_time=ZIP_TIMESTAMP)\n    info.compress_type = zipfile.ZIP_STORED\n    info.create_system = 3\n    info.external_attr = 0o100644 << 16\n    info.flag_bits = 0x800\n    return info\n\n\ndef _validate_config(payload: bytes) -> None:\n    config = json.loads(payload)\n    required = {\n        "patch_size": 16,\n        "hidden_size": 384,\n        "num_register_tokens": 4,\n    }\n    for key, expected in required.items():\n        if int(config.get(key, -1)) != expected:\n            raise ValueError(f"official DINO config {key} drifted")\n    if "use_gated_mlp" in config and not bool(config["use_gated_mlp"]):\n        raise ValueError("official S+ config no longer enables its gated MLP")\n\n\ndef verify_snapshot(snapshot: str | Path) -> dict[str, dict[str, Any]]:\n    root = Path(snapshot).resolve()\n    if not root.is_dir():\n        raise ValueError(f"snapshot directory does not exist: {root}")\n    observed: dict[str, dict[str, Any]] = {}\n    for name, expected in EXPECTED_MODEL_FILES.items():\n        path = root / name\n        if not path.is_file():\n            raise ValueError(f"official snapshot omitted {name}")\n        size = path.stat().st_size\n        digest = sha256_file(path)\n        if size != expected["size_bytes"]:\n            raise ValueError(f"{name} size drifted: {size}")\n        if digest != expected["sha256"]:\n            raise ValueError(f"{name} SHA-256 drifted: {digest}")\n        observed[name] = {"size_bytes": size, "sha256": digest}\n    _validate_config((root / "config.json").read_bytes())\n    return observed\n\n\ndef download_snapshot(*, token: str, cache_dir: str | Path) -> Path:\n    """Download only the frozen file set from the exact gated Hub revision."""\n\n    if not token.strip():\n        raise ValueError("HF_TOKEN is empty")\n    try:\n        from huggingface_hub import HfApi, snapshot_download\n    except ImportError as exc:  # pragma: no cover - exercised in Kaggle, not unit tests\n        raise RuntimeError("install huggingface_hub==0.34.4 for the download handoff") from exc\n\n    info = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION, token=token)\n    if info.sha != MODEL_REVISION:\n        raise ValueError(f"resolved model revision drifted: {info.sha}")\n    snapshot = Path(\n        snapshot_download(\n            repo_id=MODEL_ID,\n            revision=MODEL_REVISION,\n            token=token,\n            cache_dir=Path(cache_dir),\n            allow_patterns=sorted(EXPECTED_MODEL_FILES),\n        )\n    )\n    verify_snapshot(snapshot)\n    return snapshot\n\n\ndef seal_snapshot(snapshot: str | Path, output_path: str | Path) -> dict[str, Any]:\n    """Create a fixed-topology, fixed-metadata, non-overwriting transport ZIP."""\n\n    root = Path(snapshot).resolve()\n    output = Path(output_path).resolve()\n    if output.exists():\n        raise FileExistsError(f"refusing to overwrite backbone bundle: {output}")\n    output.parent.mkdir(parents=True, exist_ok=True)\n    files = verify_snapshot(root)\n    manifest: dict[str, Any] = {\n        "schema_version": "1.0.0",\n        "status": "sealed_official_backbone",\n        "model": {\n            "repository": MODEL_ID,\n            "revision": MODEL_REVISION,\n            "frozen": True,\n            "architecture": "DINOv3 ViT-S+/16",\n        },\n        "accepted_cache_contract": {\n            "file_sha256": ACCEPTED_CACHE_BACKBONE_CONTRACT_FILE_SHA256,\n            "self_sha256": ACCEPTED_CACHE_BACKBONE_CONTRACT_SELF_SHA256,\n        },\n        "files": files,\n        "member_count_including_manifest_and_completion": len(files) + 2,\n        "zip_contract": {\n            "root": BUNDLE_ROOT,\n            "compression": "stored",\n            "timestamp": "1980-01-01T00:00:00",\n            "unix_mode": "100644",\n        },\n    }\n    manifest["manifest_sha256_excludes_self"] = self_hash(\n        manifest, "manifest_sha256_excludes_self"\n    )\n    manifest_payload = json_bytes(manifest)\n    completion = {\n        "schema_version": "1.0.0",\n        "status": "complete",\n        "model_revision": MODEL_REVISION,\n        "manifest_file_sha256": sha256_bytes(manifest_payload),\n        "manifest_sha256_excludes_self": manifest["manifest_sha256_excludes_self"],\n        "member_count": len(files) + 2,\n    }\n    completion_payload = json_bytes(completion)\n\n    with zipfile.ZipFile(output, "x", allowZip64=True) as archive:\n        for name in sorted(files):\n            archive.writestr(_zip_info(f"{BUNDLE_ROOT}/{name}"), (root / name).read_bytes())\n        archive.writestr(_zip_info(f"{BUNDLE_ROOT}/{MANIFEST_NAME}"), manifest_payload)\n        archive.writestr(_zip_info(f"{BUNDLE_ROOT}/{COMPLETE_NAME}"), completion_payload)\n    return verify_bundle(output)\n\n\ndef verify_bundle(bundle_path: str | Path) -> dict[str, Any]:\n    """Independently verify a returned backbone transport bundle in place."""\n\n    bundle = Path(bundle_path).resolve()\n    if not bundle.is_file():\n        raise ValueError(f"backbone bundle does not exist: {bundle}")\n    expected_names = {\n        *(f"{BUNDLE_ROOT}/{name}" for name in EXPECTED_MODEL_FILES),\n        f"{BUNDLE_ROOT}/{MANIFEST_NAME}",\n        f"{BUNDLE_ROOT}/{COMPLETE_NAME}",\n    }\n    with zipfile.ZipFile(bundle) as archive:\n        names = archive.namelist()\n        if len(names) != len(set(names)):\n            raise ValueError("backbone bundle contains duplicate members")\n        if set(names) != expected_names:\n            raise ValueError("backbone bundle member set drifted")\n        for info in archive.infolist():\n            if info.is_dir() or info.filename.startswith(("/", "\\\\")) or ".." in Path(\n                info.filename\n            ).parts:\n                raise ValueError(f"unsafe backbone bundle member: {info.filename}")\n            if info.date_time != ZIP_TIMESTAMP or info.compress_type != zipfile.ZIP_STORED:\n                raise ValueError(f"non-deterministic ZIP metadata: {info.filename}")\n            if (info.external_attr >> 16) != 0o100644:\n                raise ValueError(f"unexpected ZIP member mode: {info.filename}")\n        bad_crc = archive.testzip()\n        if bad_crc is not None:\n            raise ValueError(f"backbone bundle CRC failure: {bad_crc}")\n\n        manifest_payload = archive.read(f"{BUNDLE_ROOT}/{MANIFEST_NAME}")\n        manifest = json.loads(manifest_payload)\n        if manifest.get("manifest_sha256_excludes_self") != self_hash(\n            manifest, "manifest_sha256_excludes_self"\n        ):\n            raise ValueError("backbone manifest self-hash mismatch")\n        if manifest.get("files") != EXPECTED_MODEL_FILES:\n            raise ValueError("backbone manifest file contract drifted")\n        if manifest.get("model", {}).get("revision") != MODEL_REVISION:\n            raise ValueError("backbone manifest revision drifted")\n        completion = json.loads(archive.read(f"{BUNDLE_ROOT}/{COMPLETE_NAME}"))\n        if completion.get("status") != "complete":\n            raise ValueError("backbone completion marker is not complete")\n        if completion.get("manifest_file_sha256") != sha256_bytes(manifest_payload):\n            raise ValueError("backbone completion marker manifest hash mismatch")\n        if completion.get("manifest_sha256_excludes_self") != manifest.get(\n            "manifest_sha256_excludes_self"\n        ):\n            raise ValueError("backbone completion marker self-hash mismatch")\n        if int(completion.get("member_count", -1)) != len(expected_names):\n            raise ValueError("backbone completion marker member count mismatch")\n\n        for name, expected in EXPECTED_MODEL_FILES.items():\n            payload = archive.read(f"{BUNDLE_ROOT}/{name}")\n            if len(payload) != expected["size_bytes"] or sha256_bytes(payload) != expected[\n                "sha256"\n            ]:\n                raise ValueError(f"sealed {name} identity mismatch")\n        _validate_config(archive.read(f"{BUNDLE_ROOT}/config.json"))\n\n    return {\n        "schema_version": "1.0.0",\n        "status": "verified",\n        "bundle_path": str(bundle),\n        "bundle_size_bytes": bundle.stat().st_size,\n        "bundle_sha256": sha256_file(bundle),\n        "model_repository": MODEL_ID,\n        "model_revision": MODEL_REVISION,\n        "weights_size_bytes": EXPECTED_MODEL_FILES["model.safetensors"]["size_bytes"],\n        "weights_sha256": EXPECTED_MODEL_FILES["model.safetensors"]["sha256"],\n        "member_count": len(expected_names),\n        "manifest_file_sha256": sha256_bytes(manifest_payload),\n        "manifest_sha256_excludes_self": manifest["manifest_sha256_excludes_self"],\n    }\n\n'
namespace = {'__name__': 'livifuser_backbone_handoff_embedded'}
exec(compile(CORE_SOURCE, 'backbone_handoff.py', 'exec'), namespace)


In [ ]:
from pathlib import Path

work = Path('/kaggle/working/livifuser_dinov3_splus_backbone_v1')
cache = work / 'hf_cache'
out = work / 'output'
out.mkdir(parents=True, exist_ok=True)
snapshot = namespace['download_snapshot'](token=HF_TOKEN, cache_dir=cache)
bundle = out / namespace['BUNDLE_FILENAME']
report = namespace['seal_snapshot'](snapshot, bundle)
del HF_TOKEN
print('BUNDLE', bundle)
print('SIZE', report['bundle_size_bytes'])
print('SHA256', report['bundle_sha256'])
print('WEIGHTS_SHA256', report['weights_sha256'])
display(bundle)
